# TrappyTV Examplar notebook

In [ ]:
# Import Packages
import os
import sys
import numpy as np
import pandas as pd

In [ ]:
## Import Bokeh module and TrappyTV class
from bokeh.io import output_notebook
output_notebook()

from trappytv import TrappyTV

## Import Data

In [ ]:
datapaths = ["data/incubation_times_analysis_data/02122025.hd5"]

In [ ]:
with pd.HDFStore(datapaths[0]) as store:
    print(store.keys())

In [ ]:
class CellView: ## Datastructure to import 
    def __init__(self, datapath, xycols=["x_unrefined", "y_unrefined"]):
        if datapath.endswith(".hd5"):
            print("HDF datastore mode!")
            self.scopeid = "Trappy-Scope"
            self.paths = {"tracks": datapath}
        else:
            self.scopeid = os.path.basename(datapath)[:2]
            self.postprocess_path = os.path.join(datapath, "postprocess")
            self.paths = {"tracks": os.path.join(self.postprocess_path, "merged_tracks.hd5"),
                        "xyr_df": os.path.join(self.postprocess_path, "xyr.hd5")
                        }
            self.first_frames = np.load(os.path.join(self.postprocess_path, "first_frames.npy"))

        self.hd5keys = None
        with pd.HDFStore(datapaths[0]) as store:
            self.hd5keys = [key.lstrip("/") for key in store.keys()]
        if "df" in self.hd5keys:
            self.dfs = {key: pd.read_hdf(value, key="df") for key, value in self.paths.items()}
        else:
            self.dfs = {key: pd.read_hdf(value, key="tracks") for key, value in self.paths.items()}
        try:
            if "metadata" or "meta" in self.hd5keys:
                key = "metadata"
                if "meta" in metadata:
                    key = "meta"
                self.metadata = pd.read_hdf(self.paths["tracks"], key=key)
            else:
                self.metadata = pd.read_hdf(self.paths["tracks"], key=key)
            if "/tracks/meta/" in self.hd5keys:
               self.metadata = pd.read_hdf(self.paths["tracks"], key="/tracks/meta/") 
        except Exception as e:
            print(e)
            print("[WARNING] Metadata not found!")

        try:
            self.dfs["xyr_df"] = pd.read_hdf(self.paths["tracks"], key="xyr_df")
        except Exception as e:
            print(e)
            print("[WARNING] XYR data msiing")
        
        if "speed" not in self.dfs["tracks"].columns:
            self.dfs["tracks"]["speed"] = np.hypot(np.gradient(self.dfs["tracks"][xycols[0]]), np.gradient(self.dfs["tracks"][xycols[1]]))
        self.stuff = {}
    def __call__(self):
        return self.dfs["tracks"]

In [ ]:
cell = CellView(datapaths[0], xycols=["x", "y"])
cell()

## View Data

In [ ]:
#%%time
TrappyTV(cell, width=800, height=800).view_ensamble(smooth_window=100)
